# 📊 Project Dataset Explorer
Load your tagged CSV, then run cells top-to-bottom. Every section is self-contained — skip any you don't care about.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
ACCENT   = '#5B8DEF'   # blue for primary bars
ACCENT2  = '#F4845F'   # orange for secondary / contrast
BG       = '#F7F9FC'

# ── Config — adjust these two paths ──────────────────────────────────────
CSV_PATH      = 'tagging_state/tagged_projects.csv'   # your output CSV
TAXONOMY_PATH = 'taxonomy.json'                        # your taxonomy JSON

# ── Load ──────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
with open(TAXONOMY_PATH, encoding='utf-8') as f:
    tx = json.load(f)

# Column groups read directly from taxonomy (single source of truth)
FEATURES   = tx['features']['values']
APP_TYPES  = tx['application_types']['values']
TECH_MODS  = tx['technology_modifiers']['values']
NUM_MODS   = list(tx['numeric_modifiers']['fields'].keys())
BOOL_MODS  = list(tx['boolean_modifiers']['fields'].keys())
ENUM_MODS  = list(tx['enum_modifiers']['fields'].keys())

df['created_at'] = pd.to_datetime(df['created_at'], utc=True, errors='coerce')

print(f'✅  Loaded {len(df):,} projects  ·  {df.shape[1]} columns')
print(f'    Date range: {df.created_at.min().date()} → {df.created_at.max().date()}')
print(f'    Organizations: {sorted(df.organization.unique())}')
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'taxonomy.json'

: 

## 1 · Dataset at a glance

In [ ]:
# Quick overview table
overview = pd.DataFrame({
    'Total projects':        [len(df)],
    'Unique categories':     [df['category'].nunique()],
    'Organizations':         [df['organization'].nunique()],
    'Feature columns':       [len(FEATURES)],
    'App-type columns':      [len(APP_TYPES)],
    'Bool modifier cols':    [len(BOOL_MODS)],
    'Enum modifier cols':    [len(ENUM_MODS)],
    'Numeric modifier cols': [len(NUM_MODS)],
}).T.rename(columns={0: 'count'})
print(overview.to_string())

# Projects per category (top 20)
cat_counts = df['category'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(10, 5))
cat_counts.plot(kind='barh', ax=ax, color=ACCENT)
ax.set_title('Projects per Category (top 20)')
ax.set_xlabel('# projects')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

# Projects over time (monthly)
monthly = df.set_index('created_at').resample('ME').size()
fig, ax = plt.subplots(figsize=(10, 3))
monthly.plot(ax=ax, color=ACCENT, linewidth=2)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color=ACCENT)
ax.set_title('Projects Scraped per Month')
ax.set_ylabel('# projects')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout(); plt.show()

## 2 · Feature prevalence & intensity
How often each feature appears (any score ≥1) and how strongly it's scored when present.

In [ ]:
feat_df   = df[FEATURES].copy()
presence  = (feat_df > 0).mean().sort_values(ascending=False)
mean_when = feat_df.replace(0, np.nan).mean().reindex(presence.index)

TOP_N = 30
top   = presence.head(TOP_N)
mw    = mean_when.reindex(top.index)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: prevalence
ax = axes[0]
bars = ax.barh(top.index[::-1], top.values[::-1], color=ACCENT)
ax.set_title(f'Feature Prevalence — top {TOP_N}\n(share of projects where score ≥ 1)')
ax.set_xlabel('fraction of projects')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
for bar, v in zip(bars, top.values[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{v:.0%}', va='center', fontsize=8)

# Right: mean score when present (1–5)
ax = axes[1]
cmap = LinearSegmentedColormap.from_list('ints', ['#b8d4f5', ACCENT, '#1a3a6e'])
colors = [cmap((v - 1) / 4) for v in mw.values[::-1]]
ax.barh(top.index[::-1], mw.values[::-1], color=colors)
ax.set_title(f'Mean Score When Present — top {TOP_N}\n(scale 1–5)')
ax.set_xlabel('mean confidence score')
ax.set_xlim(0, 5)
for i, v in enumerate(mw.values[::-1]):
    ax.text(v + 0.05, i, f'{v:.2f}', va='center', fontsize=8)

plt.tight_layout(); plt.show()

# Rarest features (potential data gaps or very niche capabilities)
print('\n🔻 Rarest features (< 2% prevalence):')
print(presence[presence < 0.02].to_string())

## 3 · Application type distribution

In [ ]:
app_df      = df[APP_TYPES]
app_prev    = app_df.mean().sort_values(ascending=False)
app_nonzero = app_prev[app_prev > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax = axes[0]
app_nonzero.plot(kind='bar', ax=ax, color=ACCENT2)
ax.set_title('Application Type Prevalence')
ax.set_ylabel('fraction of projects')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.tick_params(axis='x', rotation=45)

# How many app types per project?
ax = axes[1]
multi = app_df.sum(axis=1).value_counts().sort_index()
multi.plot(kind='bar', ax=ax, color=ACCENT)
ax.set_title('How Many App Types per Project?')
ax.set_xlabel('# types tagged')
ax.set_ylabel('# projects')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.show()

# App type co-occurrence heatmap (which types appear together?)
at = df[APP_TYPES].loc[:, app_prev > 0.01]  # only types with >1% prevalence
co = at.T.dot(at)
np.fill_diagonal(co.values, 0)  # remove self-co-occurrence
if co.max().max() > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(co, annot=True, fmt='d', cmap='Blues', ax=ax,
                linewidths=0.5, cbar_kws={'label': '# projects sharing both types'})
    ax.set_title('App Type Co-occurrence (how often two types appear on the same project)')
    plt.tight_layout(); plt.show()

## 4 · Enum modifiers distribution
project_type, ai_level, business_logic, data_volume, integration_complexity, ui_complexity, algorithmic_complexity

In [ ]:
n_cols = 3
n_rows = -(-len(ENUM_MODS) // n_cols)  # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

order_map = {
    'project_type':          ['new','maintenance','migration','bugfix','optimization','testing','consulting'],
    'ai_level':              ['none','basic','advanced'],
    'business_logic':        ['low','medium','high'],
    'data_volume':           ['small','medium','large'],
    'integration_complexity':['low','medium','high'],
    'ui_complexity':         ['low','medium','high'],
    'algorithmic_complexity':['low','medium','high'],
}
palette = {'low':'#b8d4f5','medium': ACCENT,'high':'#1a3a6e',
           'none':'#e8e8e8','basic':ACCENT,'advanced':'#1a3a6e',
           'new':ACCENT,'maintenance':ACCENT2,'migration':'#8EBD6B',
           'bugfix':'#E8C55A','optimization':'#9B7FD4','testing':'#5BBDB8','consulting':'#F47C7C'}

for i, col in enumerate(ENUM_MODS):
    ax = axes[i]
    order = order_map.get(col, sorted(df[col].dropna().unique()))
    counts = df[col].value_counts().reindex(order).fillna(0)
    colors = [palette.get(v, ACCENT) for v in counts.index]
    counts.plot(kind='bar', ax=ax, color=colors)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('# projects')
    ax.tick_params(axis='x', rotation=30)
    total = counts.sum()
    for bar, v in zip(ax.patches, counts.values):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total * 0.01,
                    f'{v/total:.0%}', ha='center', fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Enum Modifier Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 5 · Boolean modifiers

In [ ]:
bool_df   = df[BOOL_MODS]
bool_prev = bool_df.mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [ACCENT if v >= 0.3 else '#c5d9f5' for v in bool_prev.values]
bool_prev.plot(kind='bar', ax=ax, color=colors)
ax.axhline(0.3, color=ACCENT2, linestyle='--', linewidth=1, label='30% threshold')
ax.set_title('Boolean Modifier Prevalence\n(share of projects tagged 1)')
ax.set_ylabel('fraction of projects')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.tick_params(axis='x', rotation=35)
ax.legend()
for bar, v in zip(ax.patches, bool_prev.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{v:.0%}', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

## 6 · Numeric modifiers distribution
external_api_count, estimated_screens, estimated_entities

In [ ]:
fig, axes = plt.subplots(1, len(NUM_MODS), figsize=(5 * len(NUM_MODS), 4))
if len(NUM_MODS) == 1:
    axes = [axes]

for ax, col in zip(axes, NUM_MODS):
    data = df[col].dropna()
    p95  = data.quantile(0.95)
    clipped = data.clip(upper=p95)
    ax.hist(clipped, bins=20, color=ACCENT, edgecolor='white')
    ax.axvline(data.median(), color=ACCENT2, linewidth=2, label=f'median={data.median():.1f}')
    ax.axvline(data.mean(),   color='gray',  linewidth=1.5, linestyle='--', label=f'mean={data.mean():.1f}')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('value (clipped at p95)')
    ax.set_ylabel('# projects')
    ax.legend(fontsize=8)

fig.suptitle('Numeric Modifier Distributions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(df[NUM_MODS].describe().round(2))

## 7 · Feature correlation heatmap
Which features tend to appear together? High correlation = they're often requested in the same projects.

In [ ]:
# Only include features that appear in >5% of projects (otherwise correlation is noise)
common_feats = [f for f in FEATURES if (df[f] > 0).mean() > 0.05]
corr = df[common_feats].corr(method='spearman')

fig, ax = plt.subplots(figsize=(max(12, len(common_feats) * 0.4),
                                max(10, len(common_feats) * 0.4)))
mask = np.triu(np.ones_like(corr, dtype=bool))  # upper triangle
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Spearman ρ', 'shrink': 0.6})
ax.set_title('Feature Correlation (features appearing in >5% of projects)', pad=12)
plt.tight_layout(); plt.show()

# Top 15 strongest positive correlations
corr_unstacked = (corr.where(~mask).stack()
                      .reset_index()
                      .rename(columns={'level_0':'A','level_1':'B',0:'rho'})
                      .sort_values('rho', ascending=False))
print('\n🔗 Top 15 most correlated feature pairs:')
print(corr_unstacked.head(15).to_string(index=False))

## 8 · Feature score distribution (0–5)
For each top feature: how are the confidence scores distributed across the five levels?

In [ ]:
TOP_FEATURES = (df[FEATURES] > 0).mean().sort_values(ascending=False).head(16).index.tolist()

n_cols = 4
n_rows = -(-len(TOP_FEATURES) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.5 * n_rows))
axes = axes.flatten()

score_colors = ['#e8e8e8','#b8d4f5','#7aaae8',ACCENT,'#1e5dbf','#0d2d6b']

for i, feat in enumerate(TOP_FEATURES):
    ax = axes[i]
    counts = df[feat].value_counts().sort_index().reindex(range(6), fill_value=0)
    counts.drop(0, inplace=True)  # exclude 0 (absent) to focus on intensity among present
    colors = [score_colors[s] for s in counts.index]
    counts.plot(kind='bar', ax=ax, color=colors)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('confidence score')
    ax.set_ylabel('# projects')
    ax.tick_params(axis='x', rotation=0)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Score Distribution for Top 16 Features (0s excluded — shows only projects where feature is present)',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 9 · Project complexity profile
A composite complexity score combining feature count, numeric modifiers, and enum ratings.

In [ ]:
# Complexity score: sum of all feature scores + weighted numeric modifiers
df['_feature_score']  = df[FEATURES].sum(axis=1)
df['_feature_count']  = (df[FEATURES] > 0).sum(axis=1)
df['_complexity']     = (
    df['_feature_score'] * 0.5 +
    df['estimated_screens'].fillna(0) * 0.3 +
    df['estimated_entities'].fillna(0) * 0.5 +
    df['external_api_count'].fillna(0) * 1.0
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['_feature_count'], bins=20, color=ACCENT, edgecolor='white')
axes[0].axvline(df['_feature_count'].median(), color=ACCENT2, linewidth=2)
axes[0].set_title('Features Tagged per Project')
axes[0].set_xlabel('# distinct features (score ≥ 1)')

axes[1].hist(df['_feature_score'], bins=25, color=ACCENT2, edgecolor='white')
axes[1].axvline(df['_feature_score'].median(), color=ACCENT, linewidth=2)
axes[1].set_title('Total Feature Score per Project')
axes[1].set_xlabel('sum of all feature scores')

axes[2].hist(df['_complexity'], bins=25, color='#8EBD6B', edgecolor='white')
axes[2].axvline(df['_complexity'].median(), color=ACCENT2, linewidth=2)
axes[2].set_title('Composite Complexity Score')
axes[2].set_xlabel('composite score')

for ax in axes:
    ax.set_ylabel('# projects')

fig.suptitle('Project Complexity', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Complexity by category
cat_complexity = (df.groupby('category')['_complexity']
                    .median()
                    .sort_values(ascending=False)
                    .head(20))
fig, ax = plt.subplots(figsize=(10, 5))
cat_complexity.plot(kind='barh', ax=ax, color=ACCENT)
ax.invert_yaxis()
ax.set_title('Median Complexity Score by Category (top 20)')
ax.set_xlabel('median composite complexity score')
plt.tight_layout(); plt.show()

## 10 · Per-organization breakdown
How do Karlancer vs Ponisha (or whichever orgs you have) differ in feature demand?

In [ ]:
orgs = sorted(df['organization'].dropna().unique())
if len(orgs) < 2:
    print('Only one organization in dataset — skipping comparison.')
else:
    # Feature prevalence per org
    org_feat = {org: (df[df.organization == org][FEATURES] > 0).mean() for org in orgs}
    org_feat_df = pd.DataFrame(org_feat)

    # Top features with biggest gap between orgs (absolute diff)
    org_feat_df['diff'] = (org_feat_df.iloc[:, 0] - org_feat_df.iloc[:, 1]).abs()
    top_gap = org_feat_df.nlargest(15, 'diff').drop(columns='diff')

    ax = top_gap.plot(kind='barh', figsize=(12, 6),
                      color=[ACCENT, ACCENT2][:len(orgs)])
    ax.invert_yaxis()
    ax.set_title('Top 15 Feature Gaps Between Organizations')
    ax.set_xlabel('prevalence (fraction of projects)')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.legend(title='Organization')
    plt.tight_layout(); plt.show()

    # Enum modifier comparison
    for col in ['business_logic', 'ui_complexity', 'project_type']:
        pivot = (df.groupby(['organization', col])
                   .size()
                   .unstack(fill_value=0)
                   .apply(lambda r: r / r.sum(), axis=1))
        pivot.plot(kind='bar', figsize=(8, 4), colormap='Blues')
        plt.title(f'{col.replace("_"," ").title()} by Organization')
        plt.ylabel('fraction of projects')
        plt.xticks(rotation=0)
        plt.tight_layout(); plt.show()

## 11 · Feature presence matrix (sample)
A visual fingerprint of 50 random projects — each row is a project, each column is a feature.

In [ ]:
SAMPLE_N   = min(50, len(df))
sample     = df.sample(SAMPLE_N, random_state=42)
# Only include features present in at least one sampled project
feat_subset = [f for f in FEATURES if (sample[f] > 0).any()]

mat = sample[feat_subset].values
fig, ax = plt.subplots(figsize=(max(14, len(feat_subset) * 0.18), max(8, SAMPLE_N * 0.18)))
im = ax.imshow(mat, aspect='auto', cmap='Blues', vmin=0, vmax=5)
ax.set_xticks(range(len(feat_subset)))
ax.set_xticklabels(feat_subset, rotation=90, fontsize=7)
ax.set_yticks(range(SAMPLE_N))
ax.set_yticklabels(sample['title'].str[:40].tolist(), fontsize=6)
plt.colorbar(im, ax=ax, label='confidence score (0–5)')
ax.set_title(f'Feature Matrix — {SAMPLE_N} random projects (darker = higher confidence)', pad=10)
plt.tight_layout(); plt.show()

## 12 · Quick data-quality checks
Things worth reviewing before using this data for modelling.

In [ ]:
print('=== Data Quality Report ===')
print()

# Projects with zero features tagged (possible tagging failure)
zero_feat = (df[FEATURES].sum(axis=1) == 0).sum()
print(f'⚠️  Projects with NO features tagged: {zero_feat} ({zero_feat/len(df):.1%})')
if zero_feat > 0:
    print(df[df[FEATURES].sum(axis=1) == 0][['title','category','organization']].head(10).to_string())

print()

# Projects with no app type tagged
zero_app = (df[APP_TYPES].sum(axis=1) == 0).sum()
print(f'⚠️  Projects with NO application type tagged: {zero_app} ({zero_app/len(df):.1%})')

print()

# Empty enum values
for col in ENUM_MODS:
    blank = df[col].isna().sum() + (df[col] == '').sum()
    if blank > 0:
        print(f'⚠️  {col}: {blank} blank/null values')

print()

# Features always at max score (possible over-tagging)
always_max = [f for f in FEATURES if df[f].max() == 5 and (df[f] == 5).mean() > 0.5]
if always_max:
    print(f'⚠️  Features scored 5 in >50% of projects (possible over-tagging): {always_max}')

print()
print('=== Score Distribution Summary (features only) ===')
score_dist = df[FEATURES].stack().value_counts().sort_index()
total_cells = len(df) * len(FEATURES)
for score, count in score_dist.items():
    bar = '█' * int(count / total_cells * 200)
    print(f'  Score {int(score)}: {count:7,}  ({count/total_cells:.1%})  {bar}')